# Improved on real data

### Importing packages

In [ ]:
import time
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as e:
    raise RuntimeError("This notebook loads data from Google Drive; use Colab or mount Drive yourself.") from e

DATA = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed")
ask_path = DATA / "eurusd_dukascopy_ask_201901.parquet"
bid_path = DATA / "eurusd_dukascopy_bid_201901.parquet"
for p in (ask_path, bid_path):
    if not p.is_file():
        raise FileNotFoundError(p)

print("1. Loading Data...")
t0 = time.time()

bids = pd.read_parquet(bid_path)
bids["datetime"] = pd.to_datetime(bids["datetime"])
bids = bids.sort_values("datetime")
bids = bids.rename(columns={"price": "bid", "volume": "bid_vol"})

asks = pd.read_parquet(ask_path)
asks["datetime"] = pd.to_datetime(asks["datetime"])
asks = asks.sort_values("datetime")
asks = asks.rename(columns={"price": "ask", "volume": "ask_vol"})

print("2. Aligning Asynchronous Ticks (As-Of Merge)...")
quotes = pd.merge_asof(
    bids[['datetime', 'bid', 'bid_vol']], 
    asks[['datetime', 'ask']], 
    on='datetime', 
    direction='backward'
)

quotes['mid_price'] = (quotes['bid'] + quotes['ask']) / 2
quotes.dropna(subset=['mid_price'], inplace=True)
quotes.set_index('datetime', inplace=True)

t1 = time.time()
print(f"Loaded and aligned {len(quotes):,} raw ticks in {t1-t0:.2f} seconds.")
print("\n--- Mid-Price Statistics ---")
print(quotes['mid_price'].describe())

### Resampling

In [ ]:
print("3. Applying Time Bars and 'Clipping'...")
t0 = time.time()

# 1. Resample continuous dataset
time_bars = quotes['mid_price'].resample('5min').last().ffill()

# 2. Calculate returns 
returns = np.log(time_bars / time_bars.shift(1)).dropna() * 10000

# 3. Clip the hours
start_time = '10:00'
end_time = '19:55' 
returns_clipped = returns.between_time(start_time, end_time)

t1 = time.time()
print(f"Time Bar conversion took {t1-t0:.3f} seconds.")
print(f"Continuous 5-min bars: {len(returns):,}")
print(f"Clipped 5-min bars (Trading Hours Only): {len(returns_clipped):,}")
print(f"Percentage of noise filtered out: {(1 - len(returns_clipped)/len(returns))*100:.1f}%")

### GPU matrix reshaping

In [ ]:
print("4. Reshaping into [Days, Bars] Matrix for JAX...")
df_returns = pd.DataFrame({'return': returns_clipped})
df_returns['date'] = df_returns.index.date

expected_bars_per_day = 120 
valid_days = []
valid_dates = [] # We save the dates so we can plot them later!

for date, group in df_returns.groupby('date'):
    if len(group) == expected_bars_per_day:
        valid_days.append(group['return'].values)
        valid_dates.append(date)

matrix_data = np.array(valid_days)

import jax.numpy as jnp
y_jax = jnp.array(matrix_data)

print(f"Data ready for H100! Matrix shape: {y_jax.shape} (Days, Bars)")
print(f"Successfully captured {len(valid_dates)} full trading days.")

### Imports

In [6]:
%%capture
%pip install numpyro
%pip install --upgrade nvidia-cuda-nvcc-cu12
%pip install --upgrade "jax[cuda12]" numpyro

import jax
import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, autoguide
from jax.scipy.special import logsumexp
from jax import random
import matplotlib.pyplot as plt
import scipy.stats as stats

print("Hardware connected:", jax.devices())

### Training

In [ ]:
import time

def hmm_ar1_model(y):
    n_days, n_bars = y.shape
    n_states = 2

    transition_probs = numpyro.sample("transition_probs", dist.Dirichlet(jnp.ones((n_states, n_states))))
    
    beta0 = numpyro.sample("beta0", dist.Normal(jnp.zeros(n_states), 0.1)) 
    beta1 = numpyro.sample("beta1", dist.Normal(jnp.zeros(n_states), 0.5)) 
    sigma = numpyro.sample("sigma", dist.LogNormal(jnp.zeros(n_states), 1.0)) 
    
    init_probs = numpyro.sample("init_probs", dist.Dirichlet(jnp.ones(n_states)))

    def forward_one_day(y_day):
        log_alpha = jnp.log(init_probs) + dist.Normal(beta0, sigma).log_prob(y_day[0])
        
        def scan_fn(log_alpha_prev, t):
            y_curr = y_day[t]
            y_prev = y_day[t-1] 
            mu_t = beta0 + beta1 * y_prev
            
            log_emission = dist.Normal(mu_t, sigma).log_prob(y_curr)
            log_alpha_next = logsumexp(log_alpha_prev[:, None] + jnp.log(transition_probs), axis=0) + log_emission
            return log_alpha_next, None

        log_alpha_final, _ = jax.lax.scan(scan_fn, log_alpha, jnp.arange(1, n_bars))
        return logsumexp(log_alpha_final)

    log_likes = jax.vmap(forward_one_day)(y)
    numpyro.factor("log_likelihood", jnp.sum(log_likes))

# ==============================================================================
# SVI TRAINING LOOP WITH PROFILING
# ==============================================================================
print("\nCompiling AR(1) Model to GPU...")
t_comp_start = time.time()
optimizer = numpyro.optim.Adam(step_size=0.005)
guide = autoguide.AutoDelta(hmm_ar1_model)
svi = SVI(hmm_ar1_model, guide, optimizer, loss=Trace_ELBO())

rng_key = random.PRNGKey(42)
svi_state = svi.init(rng_key, y=y_jax)
t_comp_end = time.time()
print(f"Compilation finished in {t_comp_end - t_comp_start:.2f} seconds.")

n_steps = 3000
losses = []

print(f"\nStarting training loop for {n_steps} steps on {jax.devices()[0]}...")
t_train_start = time.time()

for i in range(n_steps):
    svi_state, loss = svi.update(svi_state, y=y_jax)
    losses.append(float(loss))
    if i % 500 == 0 or i == n_steps - 1:
        print(f"Step {i:4d} | Loss (Negative ELBO): {loss:.2f}")

t_train_end = time.time()
total_time = t_train_end - t_train_start
print(f"\nTraining Complete in {total_time:.2f} seconds! ({(total_time/n_steps)*1000:.2f} ms/step)")

# Extract Parameters
learned_params = svi.get_params(svi_state)
P = np.round(learned_params['transition_probs_auto_loc'], 4)
b0 = np.round(learned_params['beta0_auto_loc'], 4)
b1 = np.round(learned_params['beta1_auto_loc'], 4)
sig = np.round(learned_params['sigma_auto_loc'], 4)

# Calculate Stationary Distribution (Long-term probability of being in each state)
stat_0 = P[1, 0] / (P[0, 1] + P[1, 0])
stat_1 = 1 - stat_0

print("\n--- GPU Learned AR(1) Parameters ---")
print("Transition Matrix:\n", P)
print(f"Stationary Distribution: State 0: {stat_0*100:.1f}%, State 1: {stat_1*100:.1f}%")
print("\nState 0 Equations:")
print(f"  y_t = {b0[0]:.4f} + ({b1[0]:.4f}) * y_t-1 + N(0, {sig[0]:.4f}^2)")
print("\nState 1 Equations:")
print(f"  y_t = {b0[1]:.4f} + ({b1[1]:.4f}) * y_t-1 + N(0, {sig[1]:.4f}^2)")

plt.figure(figsize=(10, 4))
plt.plot(losses, color='darkorange', linewidth=2)
plt.title("SVI GPU Training Convergence - MS-AR(1)")
plt.xlabel("Iteration Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.show()

### Summary statistics

In [ ]:
# ==============================================================================
# VISUALIZING THE HIDDEN STATES
# ==============================================================================
# 1. Pick a day to visualize (e.g., the 5th valid day in our dataset)
day_idx = 4 
sample_date = valid_dates[day_idx]

# Grab the actual returns for that day from our matrix
y_day = matrix_data[day_idx]

# Grab the actual prices for that day from our original time_bars so we can plot price
# (We reconstruct the price path based on the returns to visualize it cleanly)
cumulative_returns = np.cumsum(y_day / 10000) 
simulated_price = 1.0 * np.exp(cumulative_returns) # Reconstruct a normalized price path

# 2. Run the Forward Filter to get State Probabilities
n = len(y_day)
probs = np.zeros((n, 2))
pi = np.array([0.5, 0.5]) # Initial belief

# Step t=0
p_y0 = stats.norm.pdf(y_day[0], loc=b0, scale=sig)
alpha = pi * p_y0
probs[0] = alpha / np.sum(alpha)

# Step t=1 to end (Using our AR(1) equation!)
for t in range(1, n):
    # Calculate the moving mean based on AR(1) beta1 and yesterday's return
    mu_t = b0 + b1 * y_day[t-1]
    
    # Check the probability of today's return given our state equations
    p_yt = stats.norm.pdf(y_day[t], loc=mu_t, scale=sig)
    
    # Update belief (Current Belief * Transition Matrix * Emission Prob)
    alpha = (probs[t-1] @ P) * p_yt
    probs[t] = alpha / np.sum(alpha)

# 3. Plot the Price with Regime Background Shading
state_1_probs = probs[:, 1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

# Top Plot: Normalized Price
ax1.plot(simulated_price, color='black', linewidth=1.5)
ax1.set_title(f"EUR/USD Normalized Price & HMM Regimes ({sample_date})", fontsize=14)
ax1.set_ylabel("Price")

# Color the background based on HMM confidence
ax1.fill_between(range(n), ax1.get_ylim()[0], ax1.get_ylim()[1], 
                 where=(state_1_probs > 0.5), color='red', alpha=0.2, label='State 1 (High Vol/Mean Reversion)')
ax1.fill_between(range(n), ax1.get_ylim()[0], ax1.get_ylim()[1], 
                 where=(state_1_probs <= 0.5), color='blue', alpha=0.1, label='State 0 (Quiet)')
ax1.legend(loc='upper left')

# Bottom Plot: The actual HMM Probability
ax2.plot(state_1_probs, color='purple')
ax2.axhline(0.5, color='black', linestyle='--')
ax2.set_title("Probability of being in State 1")
ax2.set_ylabel("Probability")
ax2.set_xlabel("5-Minute Bar Index (10:00 to 19:55)")
ax2.set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()